# Classical probability and quantum mechanics, side by side

**The punchline.** Classical probability theory and quantum mechanics are *the same
design*, instantiated on two different norms. Both say:

- a **state** is a vector of norm 1,
- **dynamics** is a linear map that preserves that norm,
- **composition** of two systems is the tensor product.

Classical probability takes the **1-norm** ($\sum_i \lvert v_i\rvert = 1$, entries non-negative);
quantum mechanics takes the **2-norm** ($\sum_i \lvert v_i\rvert^2 = 1$, entries complex). That
single substitution is the *only* input. Everything people call quantum weirdness in this
notebook — continuous reversibility, interference, entanglement — is a consequence of it.

And the relationship is not symmetric. Classical probability reappears *inside* quantum
mechanics exactly, as the fixed point of decoherence (Part 5). The reverse embedding
fails, in three specific places, which are Parts 1, 3 and 4.

We build a 40-line classical simulator, `csim`, whose gate kernel is *literally the same
code* as `qsim`'s, and then look for every place the two theories come apart.

### The route

- **Part 0** builds `csim`, the classical simulator, and shows that its gate kernel is the
  same source code as `qsim`'s. The only difference between the two theories is which
  matrices are allowed through the door.
- **Part 1** finds the first crack: reversible classical dynamics is *finite*. There is no
  square root of NOT. There is one in `qsim`.
- **Part 2** tries the obvious repair — take square roots of probabilities and be quantum
  for free — and watches it fail for a precise reason.
- **Part 3** is interference: two routes to an outcome that cancel instead of adding.
- **Part 4** is entanglement, and the sharpest single fact in the notebook: a whole that is
  more certain than its parts.
- **Part 5** builds the bridge back and shows the classical theory sitting exactly inside
  the quantum one.

Each part states a claim, runs it, and asserts it. Every assertion is repeated in the
closing cell, so if any of this ever stops being true the notebook fails loudly.

Background from the course: **[01 — states and gates](../01-states-and-gates.ipynb)**,
**[02 — entanglement](../02-entanglement.ipynb)**,
**[03 — Bell tests](../03-bell-tests-teleportation.ipynb)**,
**[06 — why the world looks classical](../06-decoherence.ipynb)**.

In [ ]:
import ast
import difflib
import inspect as pyinspect
import re
import textwrap
from itertools import permutations, product
from typing import Any

import matplotlib.pyplot as plt
import numpy as np

import qsim.state
from qsim import Circuit
from qsim.decoherence import dephasing_coupling
from qsim.gates import CNOT, SX, H, Rx, Ry, Rz, X

rng = np.random.default_rng(20260802)
np.set_printoptions(precision=4, suppress=True)

## Part 0 — A 40-line classical simulator

`qsim` stores the state of $n$ qubits as one NumPy array of shape `(2,) * n`, one complex
**amplitude** per bit pattern, with $\sum \lvert\text{amplitude}\rvert^2 = 1$. Squaring an
amplitude's magnitude gives the probability of that outcome — the **Born rule**.

A classical simulator of $n$ random bits stores... one NumPy array of shape `(2,) * n`,
one **probability** per bit pattern, with $\sum p_i = 1$. Same array, same axes, same
tensor-product structure: axis $k$ is bit $k$, and `p[0, 1, 1]` is the probability of the
pattern 011 exactly as `psi[0, 1, 1]` is the amplitude of $\lvert 011\rangle$.

The tensor shape is not a storage detail in either theory. Storing $2^n$ numbers in an
array of shape `(2,) * n` means that **the axes are the subsystems**: bit $k$ (or qubit
$k$) *is* axis $k$. Acting on one subsystem is then a 2×2 matrix applied along one axis;
ignoring a subsystem is a sum (classically) or a partial trace (quantum-mechanically) over
its axis; and combining two independent systems is `np.multiply.outer`, which is what the
tensor product looks like when you write it down. Both theories compose the same way, and
the array says so.

So let us write that simulator. `csim` below is deliberately a mirror of `qsim`: one
function per `qsim` function, same mechanics, same bit convention (bit 0 is the most
significant). It is not a toy standing in for classical physics — it is the whole of
finite classical probability theory, which really does fit in forty lines. The point of
writing it is to find out where the mirror cracks.

In [ ]:
# ---- csim: a classical-probability simulator, mirroring qsim function for function ----


def cstate(n: int) -> np.ndarray:
    """The n-bit state "certainly 00...0". Mirrors qsim.state.zero_state."""
    p = np.zeros((2,) * n)
    p[(0,) * n] = 1.0
    return p


def apply_stochastic(p: np.ndarray, s: np.ndarray, k: int) -> np.ndarray:
    """Apply the 2x2 column-stochastic matrix ``s`` to bit ``k``. Mirrors apply_1q."""
    # Contract s's column (input) index, axis 1, against the state's axis k: for every
    # combination of the other bits, the pair of probabilities (p_0, p_1) sitting along
    # axis k becomes s @ (p_0, p_1). tensordot puts the surviving row index first, so
    # move it back to position k.
    p = np.tensordot(s, p, axes=([1], [k]))
    return np.moveaxis(p, 0, k)


def apply_controlled_stochastic(
    p: np.ndarray, s: np.ndarray, controls: list[int], target: int
) -> np.ndarray:
    """Apply ``s`` to bit ``target``, only where every control bit is 1. Mirrors
    qsim.state.apply_controlled."""
    # Slicing away the control axes selects the sub-array where every control is 1; we
    # transform that half and leave the other half alone. That *is* the identity
    # "controlled-S = (control 0: do nothing) + (control 1: do S)".
    sl: list[Any] = [slice(None)] * p.ndim
    for c in controls:
        sl[c] = 1
    sub = p[tuple(sl)]
    # Slicing drops the control axes, so a target at axis t moves down by the number of
    # control axes in front of it.
    adjusted = target - sum(1 for c in controls if c < target)
    new_sub = apply_stochastic(sub, s, adjusted)
    out = p.copy()
    out[tuple(sl)] = new_sub
    return out


def marginal(p: np.ndarray, keep: list[int]) -> np.ndarray:
    """The distribution of the bits in ``keep``, ignoring the rest.

    The classical mirror of the partial trace: sum the probabilities over every axis you
    have decided not to look at.
    """
    drop = tuple(k for k in range(p.ndim) if k not in keep)
    return p.sum(axis=drop)


def csample(p: np.ndarray, shots: int, generator: np.random.Generator) -> np.ndarray:
    """Draw ``shots`` bit patterns, returned as integers. Mirrors inspect.sample."""
    # reshape(-1) in C order walks the last axis fastest, so index i of the flat array is
    # the bit pattern read as an integer with bit 0 most significant -- qsim's convention.
    return generator.choice(p.size, size=shots, p=p.reshape(-1))


# ---- the classical "gates": every column non-negative and summing to 1 ----
FLIP = np.array([[0.0, 1.0], [1.0, 0.0]])        # the NOT gate: a permutation
COIN = np.array([[0.5, 0.5], [0.5, 0.5]])        # randomize: the "classical Hadamard"


def LAZY(eps: float) -> np.ndarray:
    """The binary symmetric channel: flip the bit with probability ``eps``."""
    return np.array([[1.0 - eps, eps], [eps, 1.0 - eps]])


def CCOPY(p: np.ndarray, a: int, b: int) -> np.ndarray:
    """Copy bit ``a`` onto bit ``b`` (b ^= a): the classical CNOT."""
    return apply_controlled_stochastic(p, FLIP, [a], b)

### Diff the two simulators

`csim`'s `apply_stochastic` and `qsim`'s `apply_1q` were written to do different physics.
Let us check how different the code is. The cell below pulls the *source* of both
functions, strips comments and docstrings, renames the local variables to neutral names
(`psi`/`p` → `state`, `u`/`s` → `matrix`), and diffs what is left.

`ast.parse` turns source text into a syntax tree; `ast.unparse` prints it back out in a
canonical form, which is what throws away comments and formatting differences for us.

In [ ]:
def normalized_body(fn, renames: dict[str, str]) -> list[str]:
    """The body of ``fn`` as canonical source lines, with local names renamed."""
    tree = ast.parse(textwrap.dedent(pyinspect.getsource(fn)))
    func = tree.body[0]
    body = func.body  # type: ignore[attr-defined]
    # Drop a leading docstring: it parses as an expression statement holding a constant.
    if isinstance(body[0], ast.Expr) and isinstance(body[0].value, ast.Constant):
        body = body[1:]
    src = "\n".join(ast.unparse(stmt) for stmt in body)
    for old, new in renames.items():
        # \b is a word boundary, so renaming "s" does not touch "np" or "axes".
        src = re.sub(r"\b" + old + r"\b", new, src)
    return src.splitlines()


quantum_kernel = normalized_body(qsim.state.apply_1q, {"psi": "state", "u": "matrix"})
classical_kernel = normalized_body(apply_stochastic, {"p": "state", "s": "matrix"})

print("qsim.state.apply_1q          csim.apply_stochastic")
print("-" * 68)
for line in quantum_kernel:
    print(line)
print("-" * 68)
for line in classical_kernel:
    print(line)
print("-" * 68)
diff = list(difflib.unified_diff(quantum_kernel, classical_kernel, "qsim", "csim"))
print(f"diff: {len(diff)} lines")
assert diff == [], "the two kernels have drifted apart"

The diff is empty. The quantum simulator's gate kernel and the classical simulator's gate
kernel are the same three lines of NumPy.

So what *is* different? Only the admission criterion for the matrix:

| | classical (L1) | quantum (L2) |
|---|---|---|
| state | $p \in \mathbb{R}^{2^n}$, $p_i \ge 0$, $\sum_i p_i = 1$ | $\psi \in \mathbb{C}^{2^n}$, $\sum_i \lvert\psi_i\rvert^2 = 1$ |
| norm preserved | $\lVert p \rVert_1$ | $\lVert \psi \rVert_2$ |
| legal matrices | column-stochastic: $S_{ij} \ge 0$, $\sum_i S_{ij} = 1$ | unitary: $U^\dagger U = I$ |
| composition | tensor product | tensor product |
| "look at part of it" | marginal (sum out axes) | partial trace |
| basic randomizer | `COIN` | `H` |
| basic permutation | `FLIP` | `X` |

Two rows of that table are identical, three differ only in which norm is named, and one —
the third — is the whole subject of this notebook.

Both constraints say the same thing in their own currency: *legal states must map to legal
states*. Column-stochastic means "feed in a probability distribution, get a probability
distribution out": each column is where one input state goes, so it must itself be a
distribution. Unitary means "feed in a unit vector, get a unit vector out" — $U^\dagger U =
I$ says the columns are orthonormal, so lengths and angles survive.

But notice what else unitarity says, and stochasticity does not. $U^\dagger U = I$ hands
you the inverse for free: $U^{-1} = U^\dagger$, which is another unitary. Every quantum
gate is reversible, automatically, by the definition of the theory. Nothing in
"column-stochastic" implies invertibility — `COIN` sends every input to the same output and
has no inverse at all. The unitaries form a **group**; the stochastic matrices form only a
**semigroup**, a system in which things can be done but not always undone.

That difference is Part 1.

### One bit and one qubit, drawn

Before any dynamics, look at the two state *spaces*.

A probabilistic bit is described by one number, $p = P(\text{bit} = 1) \in [0, 1]$: a
line segment. A qubit's state, once you ignore the overall phase that no experiment can
see, is a point in a solid ball of radius 1 — the **Bloch ball**, whose coordinates are
the average values of the three Pauli observables $X$, $Y$, $Z$. The surface holds the
pure states ($\lvert 0\rangle$ at the north pole, $\lvert 1\rangle$ at the south,
$\lvert +\rangle$ on the equator); interior points are mixtures, and the exact centre is
"no information at all".

One convex body has dimension 1, the other dimension 3. That gap is where everything in
this notebook is going to live.

In [ ]:
fig_states = plt.figure(figsize=(10.5, 3.8))

ax_seg = fig_states.add_subplot(1, 2, 1)
ax_seg.plot([0, 1], [0, 0], color="#444", lw=2.5, zorder=1)
ax_seg.scatter([0, 1], [0, 0], s=90, color="#17797c", zorder=3)
ax_seg.scatter([0.3], [0], s=110, color="#c33b53", zorder=4)
ax_seg.annotate("bit = 0", (0, 0), textcoords="offset points", xytext=(-6, 12))
ax_seg.annotate("bit = 1", (1, 0), textcoords="offset points", xytext=(-14, 12))
ax_seg.annotate("p = 0.3\n= 0.7·(bit 0) + 0.3·(bit 1)\nthe only way to write it",
                (0.3, 0), textcoords="offset points", xytext=(-40, -46), fontsize=9,
                color="#c33b53")
ax_seg.set_xlim(-0.25, 1.25)
ax_seg.set_ylim(-0.75, 0.6)
ax_seg.axis("off")
ax_seg.set_title("a probabilistic bit: a 1-dimensional segment")

ax_ball = fig_states.add_subplot(1, 2, 2, projection="3d")
# A wireframe unit sphere: the usual spherical-polar parametrization, with u the azimuth
# and v the polar angle, evaluated on a grid via the outer products below.
u, v = np.linspace(0, 2 * np.pi, 30), np.linspace(0, np.pi, 16)
ax_ball.plot_wireframe(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)),
                       np.outer(np.ones_like(u), np.cos(v)),
                       color="#8a8f98", lw=0.4, alpha=0.55)
ax_ball.scatter([0], [0], [0], s=110, color="#c33b53")
for direction in ([0, 0, 1], [1, 0, 0], [0.577, 0.577, 0.577]):
    d = np.array(direction, dtype=float)
    ax_ball.plot(*np.array([-d, d]).T, color="#c33b53", lw=1.6)
ax_ball.text(0, 0, 1.25, r"$|0\rangle$", ha="center")
ax_ball.text(0, 0, -1.5, r"$|1\rangle$", ha="center")
ax_ball.set_title("a qubit: a 3-dimensional ball\n(centre = any diameter you like)")
ax_ball.set_axis_off()
# 3D axes leave a lot of white space by default; zoom fills the panel.
ax_ball.set_box_aspect((1, 1, 1), zoom=1.35)
fig_states.tight_layout()

Note one thing now and collect the payment in Part 4. **A point inside the segment
decomposes into endpoints in exactly one way**: $p = 0.3$ is $0.7$ of "bit 0" plus $0.3$
of "bit 1", and there is no second decomposition. So a classical mixed state carries an
unambiguous story — "it really is 0 or 1, and here is how often each".

**The centre of the ball decomposes along any diameter at all**, and all those
decompositions are equally valid: half $\lvert 0\rangle$ + half $\lvert 1\rangle$, or half
$\lvert +\rangle$ + half $\lvert -\rangle$. There is no fact about which one it "really"
is. That is a different kind of object.

### Both simulators conserve their norm

The definition of legal dynamics, checked. 1000 random column-stochastic matrices on the
classical side; 1000 random rotations on the quantum side.

In [ ]:
def random_stochastic_2x2(generator: np.random.Generator) -> np.ndarray:
    """A random column-stochastic 2x2 matrix: each column a random point on the simplex."""
    # rng.dirichlet(ones(2)) draws uniformly from the probability simplex, i.e. a random
    # pair (q, 1-q). Two of them, stacked as *columns*, is a random stochastic matrix.
    return generator.dirichlet(np.ones(2), size=2).T


p_classical = cstate(3)
worst_l1 = 0.0
for _ in range(1000):
    p_classical = apply_stochastic(p_classical, random_stochastic_2x2(rng),
                                   int(rng.integers(3)))
    worst_l1 = max(worst_l1, abs(p_classical.sum() - 1.0))
    assert p_classical.min() >= -1e-15          # probabilities never go negative

qc_norm = Circuit(name="norms", seed=1)
qubits_norm = qc_norm.alloc_many(3)
worst_l2 = 0.0
for _ in range(1000):
    Rx(qubits_norm[int(rng.integers(3))], theta=float(rng.uniform(0, 2 * np.pi)))
    Ry(qubits_norm[int(rng.integers(3))], theta=float(rng.uniform(0, 2 * np.pi)))
    worst_l2 = max(worst_l2, abs(qc_norm.inspect.norm() - 1.0))

print(f"worst |1-norm  - 1| over 1000 stochastic maps: {worst_l1:.2e}")
print(f"worst |2-norm  - 1| over 1000 unitaries:       {worst_l2:.2e}")
assert worst_l1 < 1e-12 and worst_l2 < 1e-12

## Part 1 — Reversibility: the simplex is rigid, the sphere is round

**The claim.** The only stochastic matrices whose inverse is also stochastic are the
*permutations*. So reversible classical dynamics is nothing but deterministic relabelling
of the bit patterns — a discrete, finite set of moves. The sphere, by contrast, admits a
continuous group of reversible maps: you can rotate a qubit by any angle you like, and
undo it by rotating back.

This is a theorem about norms. The linear maps that preserve the $p$-norm ball are, for
every $p \ne 2$, only the permutations-and-signs — a finite group. **Only $p = 2$ has a
continuous isometry group.** Quantum mechanics gets continuous reversible time evolution
because it picked the one exponent where the ball is round in every direction.

The geometry is easy to feel. The unit ball of the 1-norm is a diamond, with corners; the
unit ball of the $\infty$-norm is a cube, with corners; a linear map that preserves the
shape has to send corners to corners, and there are finitely many corners, so there are
finitely many such maps. The unit ball of the 2-norm is a sphere, which has no corners at
all — you can turn it by any angle and it looks the same. That is the entire content of the
theorem, and every strange thing in this notebook is downstream of it.

We check it numerically rather than proving it: sample hard, in the places where a
counterexample would have to hide, and see what comes back.

### (a) Hunting for a reversible randomizer

Sample 20,000 random stochastic matrices of each size (3×3 and 4×4) and keep the ones
whose inverse is *also* stochastic — non-negative entries, columns summing to 1. Then
measure how far each survivor is from the nearest permutation matrix.

Half the sample is drawn uniformly (Dirichlet columns); the other half is drawn
deliberately *near* permutation matrices, at perturbation sizes spread over twelve decades.
Sampling uniformly alone would be a rigged search — reversible maps are a measure-zero set,
so uniform sampling finds none and the claim would hold vacuously. Looking hard in exactly
the place where a counterexample could hide, and still finding only permutations, is the
honest experiment.

Following the implementation note: the inverse is taken with `np.linalg.pinv` (the
pseudo-inverse) and we then *check* that it really inverts, rather than trusting
`np.linalg.inv` on possibly ill-conditioned input. Tolerances: an inverse counts as
stochastic if its most negative entry is above $-10^{-9}$ and every column sums to 1
within $10^{-9}$; "is a permutation" means Frobenius distance below $10^{-6}$.

In [ ]:
def permutation_matrices(d: int) -> np.ndarray:
    """All d! permutation matrices, stacked into one array of shape (d!, d, d)."""
    # np.eye(d)[:, list(perm)] reorders the columns of the identity, which is exactly
    # what a permutation matrix is.
    return np.array([np.eye(d)[:, list(perm)] for perm in permutations(range(d))])


def distance_to_nearest_permutation(mats: np.ndarray, d: int) -> np.ndarray:
    """Frobenius distance from each matrix in a stack to the closest permutation."""
    perms = permutation_matrices(d)
    # Broadcast (n, 1, d, d) against (1, k, d, d) to get every sample-permutation pair,
    # then take the Frobenius norm over the two matrix axes and the min over permutations.
    diff = mats[:, None, :, :] - perms[None, :, :, :]
    return np.sqrt((diff ** 2).sum(axis=(2, 3))).min(axis=1)


def random_stochastic(generator: np.random.Generator, d: int, n: int) -> np.ndarray:
    """n random column-stochastic d x d matrices, Dirichlet columns."""
    return generator.dirichlet(np.ones(d), size=(n, d)).transpose(0, 2, 1)


def near_permutations(generator: np.random.Generator, d: int, n: int) -> np.ndarray:
    """n stochastic matrices sitting a log-uniform distance from a random permutation."""
    perms = permutation_matrices(d)
    base = perms[generator.integers(len(perms), size=n)]
    noise = random_stochastic(generator, d, n)
    eps = 10.0 ** generator.uniform(-12, 0, size=(n, 1, 1))
    return (1.0 - eps) * base + eps * noise     # a convex mix, so still stochastic


search = {}
for d in (3, 4):
    mats = np.concatenate([random_stochastic(rng, d, 10_000),
                           near_permutations(rng, d, 10_000)])
    inv = np.linalg.pinv(mats)                  # batched: one pseudo-inverse per matrix
    really_inverts = np.abs(mats @ inv - np.eye(d)).max(axis=(1, 2)) < 1e-9
    non_negative = inv.min(axis=(1, 2)) >= -1e-9
    columns_sum_to_one = np.abs(inv.sum(axis=1) - 1.0).max(axis=1) < 1e-9
    survivors = really_inverts & non_negative & columns_sum_to_one
    search[d] = (distance_to_nearest_permutation(mats, d), survivors)
    worst = search[d][0][survivors].max()
    print(f"{d}x{d}: {survivors.sum():5d} of {len(mats)} have a stochastic inverse; "
          f"the furthest from a permutation is {worst:.2e} away")
    assert worst < 1e-6

Two thousand-odd survivors out of the twenty thousand at each size — the near-permutation
half of the sample doing its job — and not one of them further than a few times $10^{-9}$
from an exact permutation, which is simply the tolerance we allowed leaking back out.

The histogram below puts that on a log axis, where the numbers span twelve decades. Grey is
every sample; red is the reversible ones. If reversible non-permutations existed, red bars
would appear out in the middle of the range, where the interesting matrices live.

In [ ]:
fig_rigidity, axes_rig = plt.subplots(1, 2, figsize=(10.5, 3.6), sharey=True)
bins = np.logspace(-13, 0.6, 60)
for ax, d in zip(axes_rig, (3, 4)):
    distances, survivors = search[d]
    ax.hist(np.clip(distances, 1e-13, None), bins=bins, color="#8a8f98",
            label=f"all {len(distances)} samples")
    ax.hist(np.clip(distances[survivors], 1e-13, None), bins=bins, color="#c33b53",
            label=f"reversible ({survivors.sum()})")
    ax.axvline(1e-6, color="#17797c", ls="--", lw=1.2, label="1e-6 tolerance")
    ax.set_xscale("log")
    ax.set_xlabel("distance to the nearest permutation matrix")
    ax.set_title(f"{d}x{d} stochastic matrices")
    ax.legend(fontsize=8, loc="upper left")
axes_rig[0].set_ylabel("count")
fig_rigidity.tight_layout()

The grey population spreads across the whole range; the red one — the matrices you could
run backwards — sits jammed against the left wall, at distances of order $10^{-9}$, which
is just the tolerance we allowed the search. Every reversible stochastic matrix is a
permutation.

Read that as a statement about physics: a classical reversible machine can shuffle its
possible states around, and that is *all* it can do. It cannot half-randomize and then
un-half-randomize.

### (b) Half a coin flip, and half a NOT

Ask for the square root of the coin flip: a stochastic $S$ with $S \cdot S = $ `COIN`. A
2×2 column-stochastic matrix is $\begin{pmatrix} a & b \\ 1-a & 1-b\end{pmatrix}$, two free
numbers in $[0,1]$, so we can simply sweep the whole set on a fine grid and look at the
smallest residual. Do the same for the square root of NOT.

In [ ]:
grid = np.linspace(0.0, 1.0, 401)
a_grid, b_grid = np.meshgrid(grid, grid, indexing="ij")
# Build every candidate at once: shape (401, 401, 2, 2), the last two axes being the
# matrix itself. The @ operator on such an array multiplies the last two axes pairwise,
# so all_candidates @ all_candidates squares 160,801 matrices in one go.
candidates = np.stack([np.stack([a_grid, b_grid], axis=-1),
                       np.stack([1 - a_grid, 1 - b_grid], axis=-1)], axis=-2)
squared = candidates @ candidates

residual_coin = np.sqrt(((squared - COIN) ** 2).sum(axis=(-2, -1)))
residual_flip = np.sqrt(((squared - FLIP) ** 2).sum(axis=(-2, -1)))
best_coin = np.unravel_index(residual_coin.argmin(), residual_coin.shape)
best_flip = np.unravel_index(residual_flip.argmin(), residual_flip.shape)

print(f"min ||S@S - COIN|| = {residual_coin.min():.3f}"
      f"  at a={grid[best_coin[0]]:.3f}, b={grid[best_coin[1]]:.3f}")
print(f"min ||S@S - FLIP|| = {residual_flip.min():.3f}"
      f"  at a={grid[best_flip[0]]:.3f}, b={grid[best_flip[1]]:.3f}")
print("\nCOIN @ COIN:\n", COIN @ COIN)
assert residual_flip.min() > 0.05

Two different answers, and the first one is a trap worth walking into.

**The square root of `COIN` exists, and it is `COIN` itself.** Flip a fair coin twice and
you have a fair coin: `COIN @ COIN == COIN`, so `COIN` is its own square root. But this is
a *fixed point*, not a halfway house — apply it once and you are already all the way at
uniform. There is no one-parameter family carrying the identity continuously to `COIN` and
back, because by (a) any reversible step is a permutation and permutations never leave the
corners. Randomizing classically is a cliff, not a slope.

**The square root of `FLIP` does not exist at all.** The residual bottoms out at 1.0, and
a determinant argument says why: $\det S = a - b$ for our parametrization, so
$\det(S^2) = (a-b)^2 \ge 0$, while $\det(\texttt{FLIP}) = -1$. No stochastic matrix squares
to NOT — not approximately, not at all.

Now the quantum side.

In [ ]:
# Half a NOT, three ways.

# 1. qsim ships it as a gate: SX, the square root of X. Applying it twice to each basis
#    state and comparing with X *is* the matrix identity SX @ SX == X, column by column.
for start in (0, 1):
    qc_sx = Circuit(seed=1)
    q_sx = qc_sx.alloc()
    qc_x = Circuit(seed=1)
    q_x = qc_x.alloc()
    if start == 1:
        X(q_sx)
        X(q_x)
    SX(q_sx)
    SX(q_sx)
    X(q_x)
    err_sx = np.abs(qc_sx.inspect.state_vector() - qc_x.inspect.state_vector()).max()
    print(f"|{start}>:  SX twice vs X once, max amplitude difference = {err_sx:.2e}")
    assert err_sx < 1e-12

# 2. And it is not an isolated gate but a point on a continuous path: Rx(theta) rotates
#    the qubit by any angle at all, and theta = pi/2 twice is theta = pi, which is X up to
#    an overall factor of -i. A global phase multiplies every amplitude equally, so it
#    changes no probability and no measurement -- it is not part of the state.
qc_half = Circuit(seed=1)
q_half = qc_half.alloc()
Rx(q_half, theta=np.pi / 2)
Rx(q_half, theta=np.pi / 2)
print("\nRx(pi/2) twice, starting from |0>:", qc_half.inspect.state_vector())
print("... which is |1> times the global phase -i")
assert abs(abs(qc_half.inspect.amplitude("1")) - 1.0) < 1e-12

$\sqrt{\text{NOT}}$ is worth staring at for a moment, because it has no classical reading
at all. Apply `SX` to $\lvert 0 \rangle$ and the qubit is in a superposition; apply it
again and it is *certainly* $\lvert 1\rangle$. Halfway through a bit flip is a real,
physical, perfectly definite state — one you can hold, transmit, and act on — and it is not
"the bit is 0 or 1 with some probability", because a probabilistic bit at $p = 0.5$ stays
at $p = 0.5$ when you flip it again.

This is the first strictly-quantum fact in the notebook. And it is not a special
coincidence about NOT: *every* unitary has a square root, and a smooth family of
fractional powers besides.

In [ ]:
# 3. Half of *anything* exists. Any unitary can be written U = V diag(e^{i*lam}) V^H with
#    V unitary; halve every eigenphase and you have a unitary square root.
hermitian = rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4))
hermitian = hermitian + hermitian.conj().T          # H = H^dagger by construction
# eigh diagonalizes a Hermitian matrix: H = V diag(lam) V^H with real lam and unitary V.
lam, V = np.linalg.eigh(hermitian)
# Functions of a matrix act on its eigenvalues: exp(iH) shares H's eigenvectors and has
# eigenvalues exp(i*lam). V * row_vector scales V's columns, which is V @ diag(...).
unitary = (V * np.exp(1j * lam)) @ V.conj().T
root = (V * np.exp(0.5j * lam)) @ V.conj().T
print("U unitary to      ", np.abs(unitary.conj().T @ unitary - np.eye(4)).max())
print("sqrt(U) unitary to", np.abs(root.conj().T @ root - np.eye(4)).max())
print("||sqrt(U)^2 - U|| =", np.abs(root @ root - unitary).max())
assert np.abs(root @ root - unitary).max() < 1e-12

### (c) The sweep: a crawl and an orbit

Same experiment on both sides, repeated 40 times. Classically: apply `LAZY(0.1)`, a bit
that flips with probability 0.1. Quantum-mechanically: apply `Rx(0.1)`, a rotation of the
qubit by 0.1 radians about the $x$-axis of the Bloch ball.

Both are "a small nudge, repeated". Watch what they do.

In [ ]:
STEPS = 40

p_lazy = np.array([1.0, 0.0])                    # certainly 0
lazy_track, lazy_l1 = [p_lazy.copy()], []
for _ in range(STEPS):
    lazy_l1.append(np.abs(p_lazy - 0.5).sum())   # 1-norm distance to the uniform state
    p_lazy = LAZY(0.1) @ p_lazy
    lazy_track.append(p_lazy.copy())
lazy_track = np.array(lazy_track)

qc_orbit = Circuit(name="orbit", seed=3)
q_orbit = qc_orbit.alloc()
bloch_track = [qc_orbit.inspect.bloch_vector(q_orbit)]
for _ in range(STEPS):
    Rx(q_orbit, theta=0.1)
    bloch_track.append(qc_orbit.inspect.bloch_vector(q_orbit))
bloch_track = np.array(bloch_track)

bloch_norms = np.linalg.norm(bloch_track, axis=1)
print(f"classical: P(1) went 0 -> {lazy_track[-1, 1]:.4f}; "
      f"1-norm distance to uniform monotone: {bool(np.all(np.diff(lazy_l1) <= 1e-15))}")
print(f"quantum:   Bloch vector length constant to {np.abs(bloch_norms - 1).max():.2e}")
assert np.all(np.diff(lazy_l1) <= 1e-15)
assert np.abs(bloch_norms - 1).max() < 1e-12

Both trajectories are plotted in their own state space: the classical one on the segment of
Part 0, the quantum one on a slice through the Bloch ball. Colour runs from dark (step 0)
to light (step 40) so you can see the direction of travel.

Watch for two things — whether the path ever comes back, and whether it stays on the
boundary of its state space.

In [ ]:
fig_sweep, (ax_crawl, ax_orbit) = plt.subplots(1, 2, figsize=(10.5, 4.0))

ax_crawl.plot([0, 1], [0, 0], color="#444", lw=2.0, zorder=1)
ax_crawl.scatter(lazy_track[:, 1], np.zeros(STEPS + 1), s=32,
                 c=np.arange(STEPS + 1), cmap="autumn_r", zorder=3)
ax_crawl.axvline(0.5, color="#17797c", ls="--", lw=1.2)
ax_crawl.annotate("start\nP(1) = 0", (0.0, 0), textcoords="offset points",
                  xytext=(-8, 22), fontsize=9)
ax_crawl.annotate("uniform:\nthe absorbing end", (0.5, 0), textcoords="offset points",
                  xytext=(-30, -46), fontsize=9, color="#17797c")
ax_crawl.set_xlim(-0.15, 1.05)
ax_crawl.set_ylim(-0.8, 0.8)
ax_crawl.set_yticks([])
for side in ("top", "right", "left"):
    ax_crawl.spines[side].set_visible(False)
ax_crawl.set_xlabel("P(bit = 1)")
ax_crawl.set_title("classical: 40 x LAZY(0.1) — a one-way crawl")

circle = np.linspace(0, 2 * np.pi, 200)
ax_orbit.plot(np.cos(circle), np.sin(circle), color="#8a8f98", lw=1.0)
ax_orbit.scatter(bloch_track[:, 1], bloch_track[:, 2], s=32,
                 c=np.arange(STEPS + 1), cmap="autumn_r", zorder=3)
ax_orbit.scatter([0], [0], marker="+", s=90, color="#17797c")
ax_orbit.annotate(r"start: $|0\rangle$", (0, 1), textcoords="offset points",
                  xytext=(6, -4), fontsize=9)
ax_orbit.set_xlabel("Bloch y")
ax_orbit.set_ylabel("Bloch z")
ax_orbit.set_aspect("equal")
ax_orbit.set_title("quantum: 40 x Rx(0.1) — an orbit, forever")

fig_sweep.tight_layout()

The classical state slides toward the centre of the segment and stops there. It is
monotone: the 1-norm distance to the uniform distribution never increases, at any step,
for any binary symmetric channel. Information is being destroyed, and destroyed
irreversibly, by an operation that looks like the mildest possible nudge.

The qubit goes round and round. After $2\pi/0.1 \approx 63$ steps its Bloch vector is back
exactly where it started, still of length 1 to $10^{-15}$. Nothing was lost, because there
is nowhere for it to leak: rotations of a round ball preserve the surface. (The *state
vector* takes twice as long to come home — $R_x(4\pi)$, not $R_x(2\pi)$, is the identity,
because a rotation by $\theta$ moves amplitudes by $\theta/2$. That half-angle is the
subject of **[quaternions_and_spin](quaternions_and_spin.ipynb)**, and it is invisible
here because a global sign is not part of the state.)

This is the **$p$-norm rigidity theorem** doing physics. It is also the same theorem that
answers "why squared amplitudes?" from the other direction — demand continuous reversible
dynamics on a probability-like state space and the 2-norm is forced. That argument gets its
own exhibit later in Track E (`gleason_teaser.ipynb`, not yet built).

## Part 2 — $\sqrt{p}$: the Fisher–Rao sphere, and why it has no dynamics

If the difference is 1-norm versus 2-norm, there is an obvious cheat: take square roots.
Send a probability vector $p$ to $r_i = \sqrt{p_i}$. Then $\sum_i p_i = 1$ becomes
$\sum_i r_i^2 = 1$, and the classical state is a unit vector in the 2-norm — a point on
the positive part of a sphere. Classical probability, geometrically quantum, for free.

**The claim:** this works perfectly as *geometry* and fails completely as *physics*,
because no interesting classical dynamics is linear in $\sqrt{p}$.

### (a) The embedding, and what the inner product means

The inner product of two embedded states is $\sum_i \sqrt{p_i q_i}$ — the **Bhattacharyya
coefficient**, also called the classical fidelity, a standard measure of how much two
distributions overlap. It is 1 for identical distributions and 0 for distributions that
never produce the same outcome. Orthogonal vectors ↔ distinguishable distributions, which
is exactly the role orthogonality plays in quantum mechanics.

In [ ]:
examples = {
    "fair coin vs fair coin": (np.array([0.5, 0.5]), np.array([0.5, 0.5])),
    "fair coin vs 90/10": (np.array([0.5, 0.5]), np.array([0.9, 0.1])),
    "disjoint support": (np.array([0.6, 0.4, 0.0]), np.array([0.0, 0.0, 1.0])),
    "3-outcome pair": (np.array([0.2, 0.3, 0.5]), np.array([0.1, 0.6, 0.3])),
}
worst_embed = 0.0
for label, (p_a, p_b) in examples.items():
    r_a, r_b = np.sqrt(p_a), np.sqrt(p_b)
    overlap = float(r_a @ r_b)
    bhattacharyya = float(np.sqrt(p_a * p_b).sum())
    worst_embed = max(worst_embed, abs(np.linalg.norm(r_a) - 1.0),
                      abs(overlap - bhattacharyya))
    print(f"{label:24s}  ||r_a||_2 = {np.linalg.norm(r_a):.12f}   "
          f"<r_a|r_b> = {overlap:.6f}   Bhattacharyya = {bhattacharyya:.6f}")
assert worst_embed < 1e-12

Disjoint support gives inner product exactly 0: two distributions that can never be
confused are orthogonal vectors, the same way $\lvert 0\rangle$ and $\lvert 1\rangle$ are.

### (b) The simplex becomes an octant of the sphere

For three outcomes, the classical states form a triangle (the 2-simplex). Its square-root
image is the part of the unit sphere with all three coordinates non-negative — one octant
out of eight. And that octant is a piece of the very same sphere a qubit lives on.

In [ ]:
simplex_points = rng.dirichlet(np.ones(3), size=1500)
embedded = np.sqrt(simplex_points)
print("every embedded point on the unit sphere to",
      f"{np.abs(np.linalg.norm(embedded, axis=1) - 1).max():.2e}")

fig_octant = plt.figure(figsize=(10.5, 4.2))

ax_simplex = fig_octant.add_subplot(1, 2, 1)
# Drop the 3 barycentric coordinates onto the plane: corner k at angle 90 + 120k degrees.
corners = np.array([[np.cos(a), np.sin(a)] for a in np.radians([90, 210, 330])])
flat = simplex_points @ corners
ax_simplex.scatter(flat[:, 0], flat[:, 1], s=4, color="#8a8f98", alpha=0.6)
ax_simplex.scatter(*corners.T, s=70, color="#17797c", zorder=3)
for corner, name in zip(corners, ("outcome 0", "outcome 1", "outcome 2")):
    ax_simplex.annotate(name, corner * 1.18, ha="center", fontsize=9)
ax_simplex.set_xlim(-1.45, 1.45)
ax_simplex.set_ylim(-1.35, 1.45)
ax_simplex.set_aspect("equal")
ax_simplex.axis("off")
ax_simplex.set_title("the 3-outcome simplex: a flat triangle")

ax_sphere = fig_octant.add_subplot(1, 2, 2, projection="3d")
ax_sphere.plot_wireframe(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)),
                         np.outer(np.ones_like(u), np.cos(v)),
                         color="#8a8f98", lw=0.3, alpha=0.35)
ax_sphere.scatter(embedded[:, 0], embedded[:, 1], embedded[:, 2], s=4,
                  color="#c33b53", alpha=0.75)
# The three quarter-circles bounding the octant, one in each coordinate plane. Drawing
# them makes it visible that the red patch really is one eighth of the sphere.
arc = np.linspace(0, np.pi / 2, 80)
flat_arc = np.zeros_like(arc)
for xs, ys, zs in ((np.cos(arc), np.sin(arc), flat_arc),
                   (flat_arc, np.cos(arc), np.sin(arc)),
                   (np.sin(arc), flat_arc, np.cos(arc))):
    ax_sphere.plot(xs, ys, zs, color="#17797c", lw=1.8)
ax_sphere.set_title(r"its image under $p \mapsto \sqrt{p}$:" "\none octant of the sphere")
ax_sphere.set_axis_off()
ax_sphere.view_init(elev=18, azim=28)
ax_sphere.set_box_aspect((1, 1, 1), zoom=1.35)
fig_octant.tight_layout()

The three corners of the triangle — the certain outcomes — land on the three poles where
the octant meets the axes, and the flat interior inflates onto the curved surface. Distance
along that surface is the **Fisher–Rao metric**, the natural notion of "how distinguishable
are these two distributions", and the embedding makes it the ordinary angle between two
unit vectors. This is a genuinely useful piece of mathematics, not a trick: it is why
statisticians talk about the geometry of a family of distributions at all.

And the picture is suggestive in the way this notebook is about. The qubit's own state
space is the *whole* sphere (with amplitudes complex and signs free); the classical states
sit on one eighth of it, the part where every coordinate is non-negative. Classical
probability looks like quantum mechanics confined to the positive corner.

The question is whether it can *move* there.

### (c) The failure: no linear dynamics survives the square root

Here is the test. Take a stochastic map $S$. In probability coordinates it acts linearly:
$p \mapsto Sp$. In square-root coordinates, does *any* linear map $M$ reproduce it, i.e.
is there an $M$ with $\sqrt{Sp} = M\sqrt{p}$ for all $p$?

Find the best possible $M$ by least squares over 400 sample distributions, and look at the
residual it leaves. `np.linalg.lstsq(R, T)` solves $\min_M \lVert RM - T\rVert$ where the
rows of $R$ are the inputs $\sqrt{p}$ and the rows of $T$ the targets $\sqrt{Sp}$ — a
linear regression with a matrix of unknowns.

In [ ]:
def linear_fit_residual(s: np.ndarray, generator: np.random.Generator, n: int = 400) -> float:
    """How badly the best linear map on sqrt(p) fails to reproduce p -> S p."""
    samples = generator.dirichlet(np.ones(len(s)), size=n)
    r_in = np.sqrt(samples)                       # rows: sqrt(p)
    r_out = np.sqrt(samples @ s.T)                # rows: sqrt(S p)
    best_m, *_ = np.linalg.lstsq(r_in, r_out, rcond=None)
    return float(np.linalg.norm(r_out - r_in @ best_m) / np.sqrt(n))


perm_residuals = [linear_fit_residual(perm, rng) for perm in permutation_matrices(3)]
random_residuals = [linear_fit_residual(s, rng) for s in random_stochastic(rng, 3, 200)]
coin3 = np.full((3, 3), 1 / 3)

print(f"permutations (6 of them):   residual <= {max(perm_residuals):.2e}")
print(f"random stochastic S (200):  residual >= {min(random_residuals):.4f} "
      f"(median {np.median(random_residuals):.4f})")
print(f"the 3-outcome randomizer:   residual  = {linear_fit_residual(coin3, rng):.4f}")
assert max(perm_residuals) < 1e-10
assert min(random_residuals) > 0.01

Permutations pass exactly — of course: reordering the entries of $p$ and reordering the
entries of $\sqrt{p}$ are the same operation, so a permutation *is* linear in both
coordinate systems. Everything else fails, and fails by a lot.

So the square-root trick buys the geometry and not the physics. The positive orthant of the
sphere is a statics-only theory: it has states, a notion of distance, and an inner product
with a good interpretation, but the only dynamics that is linear on it is relabelling —
Part 1's rigidity, arriving a second time by a different road.

**To get linear dynamics on amplitudes, the amplitudes have to be allowed to leave the
positive orthant.** That gives a hierarchy:

| amplitudes | theory | reversible dynamics available |
|---|---|---|
| non-negative reals | classical statics ($\sqrt{p}$ embedding) | permutations only |
| signed reals | real quantum mechanics (Stueckelberg) | rotations $O(n)$ |
| complex | quantum mechanics | unitaries $U(n)$ |

Real quantum mechanics is a genuine theory, not a strawman — and it is *experimentally
falsified* as a description of nature by a Bell-type test that real amplitudes cannot pass
(arXiv:2101.10873 and the experiments in its lineage). Cited, not developed: it would be a
notebook of its own.

Once amplitudes can be negative, two of them can cancel. That is Part 3.

## Part 3 — Two paths: addition versus cancellation

**The claim.** Classical alternatives always *add*: if an outcome can be reached two ways,
its probability is the sum of the two probabilities, and adding non-negative numbers never
gives less than either one. Amplitudes can *cancel*. This is the resource behind every
quantum algorithm.

Build the same two-stage machine in both theories. Classically: `COIN`, then `COIN`.
Quantum-mechanically: `H`, then a phase $\phi$ on the $\lvert 1\rangle$ branch (the gate
`Rz`), then `H`. That is a Mach–Zehnder interferometer — see
**[05 — interferometers](../05-interferometers.ipynb)** — with $H$ as the beam splitter.

Note that the classical machine has no place to *put* $\phi$. A phase is not an operation on
a probability; the classical apparatus has no slot for it.

Why classical alternatives can only add is worth spelling out, because it sounds like an
assumption and is actually forced. In `csim` the number attached to a route through the
machine is the *probability* of taking that route, which is a product of non-negative
factors and so is itself non-negative. The probability of arriving somewhere is the sum
over routes. A sum of non-negative numbers is at least as large as any one of them: opening
a second route to an outcome can never make that outcome rarer.

In `qsim` the number attached to a route is an *amplitude* — complex, with a sign and a
phase — and the amplitude of arriving somewhere is the sum over routes, with the
probability taken only at the very end, by squaring. Two routes with amplitudes $+\tfrac12$
and $-\tfrac12$ sum to zero. Opening a second route can make an outcome *impossible*, and
that is the whole game.

In [ ]:
phis = np.linspace(0, 2 * np.pi, 121)

classical_p0 = []
for phi in phis:
    p_interf = cstate(1)
    p_interf = apply_stochastic(p_interf, COIN, 0)
    # Whatever "insert a phase" would mean, there is nothing to insert it into.
    p_interf = apply_stochastic(p_interf, COIN, 0)
    classical_p0.append(p_interf[0])

quantum_p0 = []
for phi in phis:
    qc_interf = Circuit(name="mz", seed=11)
    q_interf = qc_interf.alloc()
    H(q_interf)
    Rz(q_interf, theta=float(phi))
    H(q_interf)
    quantum_p0.append(qc_interf.inspect.probabilities()[0])

classical_p0 = np.array(classical_p0)
quantum_p0 = np.array(quantum_p0)
print(f"classical P(0): flat at 0.5 to {np.abs(classical_p0 - 0.5).max():.2e}")
print(f"quantum P(0) vs cos^2(phi/2): max error "
      f"{np.abs(quantum_p0 - np.cos(phis / 2) ** 2).max():.2e}")
assert np.abs(classical_p0 - 0.5).max() < 1e-12
assert np.abs(quantum_p0 - np.cos(phis / 2) ** 2).max() < 1e-12

Both curves, on one pair of axes. The horizontal axis is the phase $\phi$ inserted between
the two stages — for the classical machine, a knob that is not connected to anything.

In [ ]:
fig_interference, ax_int = plt.subplots(figsize=(7.4, 4.0))
ax_int.plot(phis, quantum_p0, color="#c33b53", lw=2.4,
            label=r"quantum: $H$, $R_z(\phi)$, $H$")
ax_int.plot(phis, np.cos(phis / 2) ** 2, "k--", lw=1.1, label=r"$\cos^2(\phi/2)$")
ax_int.plot(phis, classical_p0, color="#17797c", lw=2.4, label="classical: COIN, COIN")
ax_int.axhline(0.5, color="#8a8f98", lw=0.7)
ax_int.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi],
                  ["0", "π/2", "π", "3π/2", "2π"])
ax_int.set_xlabel(r"phase $\phi$ inserted between the two beam splitters")
ax_int.set_ylabel("P(outcome 0)")
ax_int.set_ylim(-0.05, 1.05)
ax_int.set_title("two paths: one theory can cancel them, the other cannot")
ax_int.legend()
fig_interference.tight_layout()

The classical curve is flat at 0.5 and could not be anything else. Two `COIN`s in a row are
one `COIN` (`COIN @ COIN == COIN`, Part 1b), and the result does not depend on anything
you could do in between — because the only things you *can* do in between are more
stochastic matrices, which cannot un-randomize.

The quantum curve reaches 1 and 0. At $\phi = 0$ the two beam splitters *undo* each other:
$H$ is its own inverse, so $HH = I$ and the qubit is back in $\lvert 0\rangle$ with
certainty. A coin flip that un-flips.

Where did the probability of outcome 1 go? Look at the two paths individually.

In [ ]:
# Between the beam splitters, the state is (|0> + |1>)/sqrt(2) -- two live branches. The
# amplitude to end in |1> is a sum over which branch the qubit took, each term being
# (amplitude to enter the branch) x (amplitude to leave it for |1>).
qc_paths = Circuit(name="paths", seed=11)
q_paths = qc_paths.alloc()
H(q_paths)
mid = qc_paths.inspect.state_tensor()
h_matrix = np.array([[1, 1], [1, -1]]) / np.sqrt(2)

via_0 = mid[0] * h_matrix[1, 0]      # entered branch |0>, then |0> -> |1>
via_1 = mid[1] * h_matrix[1, 1]      # entered branch |1>, then |1> -> |1>
print(f"amplitude via the |0> branch: {via_0:+.4f}")
print(f"amplitude via the |1> branch: {via_1:+.4f}")
print(f"sum = {via_0 + via_1:+.4f}   ->  P(1) = {abs(via_0 + via_1) ** 2:.4f}")
print(f"if we had added *probabilities* instead: "
      f"{abs(via_0) ** 2 + abs(via_1) ** 2:.4f}")
assert abs(via_0 - 0.5) < 1e-12 and abs(via_1 + 0.5) < 1e-12

$+\tfrac12$ and $-\tfrac12$. Each path on its own would deliver the outcome a quarter of
the time; together they deliver it never. Adding the probabilities — the only thing a
classical theory can do — gives $0.25 + 0.25 = 0.5$, which is the flat teal line, and it
is *wrong* as a description of the experiment.

This closes the loop with Part 1. Interference is *why* continuous reversibility is
possible: `H` spreads the state out and a second `H` gathers it back, with the unwanted
amplitudes destroying one another instead of leaking away as lost probability. Cancellation
is what lets a randomizer be undone, and $\sqrt{\text{NOT}}$ exists for the same reason.

### An aside: cashing in the cancellation

The Elitzur–Vaidman bomb tester is three gates of payoff. A photon enters the
interferometer; a bomb, if it is live, records which arm the photon took (a `CNOT` onto a
bomb qubit). Recording destroys the cancellation — and *that is detectable*, without the
bomb going off. **[05 — interferometers](../05-interferometers.ipynb)** develops this
properly; here is just the number.

In [ ]:
qc_bomb = Circuit(name="bomb", seed=5)
photon, bomb = qc_bomb.alloc_many(2)
H(photon)
CNOT(photon, bomb)                   # a live bomb learns which arm the photon is in
H(photon)
probs_bomb = qc_bomb.inspect.probabilities()
# Bit order is (photon, bomb): index 2 = |10> = photon detected at the "impossible" port
# while the bomb has NOT absorbed it.
print("outcome probabilities |photon,bomb>:", probs_bomb)
print(f"P(bomb intact and photon at the dark port) = {probs_bomb[2]:.3f}")
print("With no bomb, HH = I and that port never fires — so a click there means a live")
print("bomb, detected by a photon that did not interact with it.")
assert abs(probs_bomb[2] - 0.25) < 1e-12

## Part 4 — Correlation versus entanglement: purity is the difference

**The claim.** The shared coin and the Bell state have *identical* statistics in the
computational basis. They differ in what "pure" means, and in what happens when you rotate
the question.

### (a) Two ways to build a perfectly correlated pair

Classically: randomize bit $a$ with `COIN`, then copy it onto bit $b$ with `CCOPY`.
Quantum-mechanically: `H` on qubit $a$, then `CNOT` from $a$ to $b$. Same shape of circuit,
same gate positions.

In [ ]:
p_pair = cstate(2)
p_pair = apply_stochastic(p_pair, COIN, 0)
p_pair = CCOPY(p_pair, 0, 1)

qc_bell = Circuit(name="bell", seed=17)
alice, bob = qc_bell.alloc_many(2)
H(alice)
CNOT(alice, bob)

print("classical joint distribution   :", p_pair.reshape(-1))
print("quantum |amplitude|^2          :", qc_bell.inspect.probabilities())
print("quantum state                  :", qc_bell.inspect.ket())
print()
print("classical marginal of bit a    :", marginal(p_pair, [0]))
print("quantum reduced density matrix of qubit a:")
print(qc_bell.inspect.reduced_density_matrix([alice]))

assert np.allclose(p_pair.reshape(-1), qc_bell.inspect.probabilities(), atol=1e-12)
assert np.allclose(marginal(p_pair, [0]), [0.5, 0.5], atol=1e-12)
assert np.allclose(qc_bell.inspect.reduced_density_matrix([alice]),
                   np.eye(2) / 2, atol=1e-12)

Identical. Both give $[0.5, 0, 0, 0.5]$: the two bits always agree, each is a fair coin on
its own. No measurement in the computational basis, at any number of shots, can tell these
two systems apart.

### (b) Purity — the single most important cell in this notebook

The classical joint state is **mixed**, and Part 0's segment tells us exactly what that
means: it has a unique decomposition into definite states.

$$[0.5,\,0,\,0,\,0.5] \;=\; 0.5\cdot(\text{both }0) \;+\; 0.5\cdot(\text{both }1)$$

and there is no other way to write it. So the classical story is forced: the bits *really
are* both 0 or both 1, we merely do not know which. Correlation is ignorance of a definite
fact.

The quantum joint state is **pure** — it is a vector, $(\lvert 00\rangle + \lvert
11\rangle)/\sqrt2$, not a mixture of anything. Full knowledge, nothing hidden. And yet
each half of it, looked at alone, is the maximally mixed state.

The measure of that is **entropy**. Entropy counts, in bits, how much you do not know.
Classically it is Shannon's $-\sum_i p_i \log_2 p_i$; quantum-mechanically it is the same
formula applied to the eigenvalues of the reduced density matrix, which `qsim` reports as
`entanglement_entropy`.

In [ ]:
def shannon(p: np.ndarray) -> float:
    """Shannon entropy in bits. Zero-probability outcomes contribute nothing."""
    p = p[p > 1e-15]                             # 0*log(0) is 0 but numpy says nan
    return float(-(p * np.log2(p)).sum())


s_joint_c = shannon(p_pair.reshape(-1))
s_marginal_c = shannon(marginal(p_pair, [0]))
s_joint_q = qc_bell.inspect.entanglement_entropy([alice, bob])
s_marginal_q = qc_bell.inspect.entanglement_entropy([alice])

print(f"classical:  S(joint) = {s_joint_c:.3f} bits   S(bit a) = {s_marginal_c:.3f} bits")
print(f"quantum:    S(joint) = {s_joint_q:.3f} bits   S(qubit a) = {s_marginal_q:.3f} bits")
print()
print(f"classical:  S(joint) >= S(part)?  {s_joint_c >= s_marginal_c - 1e-12}"
      "   (a theorem: always true)")
print(f"quantum:    S(joint) <  S(part)?  {s_joint_q < s_marginal_q - 1e-12}"
      "   (impossible classically)")

assert abs(s_joint_c - 1.0) < 1e-12 and abs(s_marginal_c - 1.0) < 1e-12
assert abs(s_joint_q) < 1e-12 and abs(s_marginal_q - 1.0) < 1e-12
assert s_joint_q < s_marginal_q - 0.5

There it is. Classically, the whole is *at least* as uncertain as any of its parts —
$S(AB) \ge S(A)$ is a theorem of probability theory, and here both sides are 1 bit.
Quantum-mechanically the whole has **zero** entropy while each part has **one bit**.

Knowing everything there is to know about the pair tells you nothing whatever about either
member. There is no classical state of affairs this can be ignorance *of*: any candidate
"really it was $\lvert 00\rangle$ or really $\lvert 11\rangle$" is a mixed state with 1 bit
of joint entropy, and the actual state has 0. The correlation is not stored in the parts,
and it is not a fact we happen to be missing. It is in the state of the pair and nowhere
else.

Entanglement is correlation in a condition of *maximal knowledge*. That is the whole
difference, and (c) and (d) turn it into numbers you could measure in a laboratory.

It is worth closing off the obvious escape route once, explicitly. "Surely the Bell pair
*is* really $\lvert 00\rangle$ or really $\lvert 11\rangle$, and the vector is just our
bookkeeping." If that were so, the pair's state would be the mixture — the classical
$[0.5, 0, 0, 0.5]$ — with 1 bit of joint entropy. It is not: the simulator holds a vector,
the entropy really is 0, and the two are different objects that happen to agree on one
question. Part 0's picture said as much in advance: the centre of the Bloch ball has no
preferred decomposition, so "which one is it really" has no answer to be ignorant of.
Part (d) will make this a measurable statement rather than an interpretive one.

### (c) Rotating the question

The computational basis is one question among many. Quantum-mechanically we can ask a
tilted one: rotate each qubit by `Ry` before measuring, which is the same experiment as
turning the measuring apparatus instead of the sample. Turn Alice's by $-\theta$ and Bob's
by $+\theta$ and measure the **correlator** $E$ = the average of (Alice's answer) ×
(Bob's answer), with answers read as $\pm 1$.

Classically there is no such move. `csim` has no rotations; the only operations on a
definite bit are stochastic matrices applied before it is read out. The most general
classical model is therefore: the pair shares some random variable $\lambda$ fixed when
they were created, and each side answers by some (possibly random) local rule that sees
$\lambda$ and its own setting. That is exactly a **local hidden-variable model**, and here
is the natural one — $\lambda$ is a shared random direction, and each side answers with the
sign of its own axis projected on it.

In [ ]:
thetas = np.linspace(0, np.pi / 2, 61)

quantum_E = []
for theta in thetas:
    qc_rot = Circuit(name="rotate", seed=17)
    a_rot, b_rot = qc_rot.alloc_many(2)
    H(a_rot)
    CNOT(a_rot, b_rot)
    Ry(a_rot, theta=float(-theta))
    Ry(b_rot, theta=float(+theta))
    # expectation("ZZ") is the average of (+1 for outcome 0, -1 for outcome 1) on each
    # qubit, multiplied together: precisely the correlator E.
    quantum_E.append(qc_rot.inspect.expectation("ZZ"))
quantum_E = np.array(quantum_E)

# The local hidden-variable model, Monte-Carlo'd honestly: one shared random direction per
# round, each side answering sign(cos(lambda - its own axis angle)).
lambdas = rng.uniform(0, 2 * np.pi, size=200_000)
lhv_E = np.array([
    np.mean(np.sign(np.cos(lambdas + theta)) * np.sign(np.cos(lambdas - theta)))
    for theta in thetas
])
# Analytically that model gives 1 - 2*(separation)/pi, a straight line in the angle.
lhv_exact = 1.0 - 2.0 * (2 * thetas) / np.pi

print(f"quantum E vs cos(2 theta):   max error {np.abs(quantum_E - np.cos(2 * thetas)).max():.2e}")
print(f"LHV sample vs its exact law: max error {np.abs(lhv_E - lhv_exact).max():.3f}")
print("quantum correlation at least as strong everywhere: "
      f"{bool(np.all(np.abs(quantum_E) >= np.abs(lhv_exact) - 1e-12))}")
print(f"and strictly stronger at theta = pi/8: |E| = {abs(quantum_E[15]):.4f} "
      f"vs {abs(lhv_exact[15]):.4f}")
assert np.abs(quantum_E - np.cos(2 * thetas)).max() < 1e-12
assert np.all(np.abs(quantum_E) >= np.abs(lhv_exact) - 1e-12)
assert abs(quantum_E[15]) > abs(lhv_exact[15]) + 0.1

The classical model was *sampled*, 200,000 shared random directions, and lands on its
exact law $E = 1 - 4\theta/\pi$ to about $0.003$ — sampling noise of order
$1/\sqrt{200000}$, as it should be. Plotting both the sample and the law is the honest
way to show that the straight line is the model's actual behaviour rather than an
assumption fed in.

In [ ]:
fig_correlator, ax_corr = plt.subplots(figsize=(7.4, 3.8))
ax_corr.plot(thetas, quantum_E, color="#c33b53", lw=2.4, label=r"quantum: $\cos 2\theta$")
ax_corr.plot(thetas, lhv_E, color="#17797c", lw=2.0,
             label="shared-randomness model (sampled)")
ax_corr.plot(thetas, lhv_exact, "k--", lw=1.0, label=r"its exact law: $1 - 4\theta/\pi$")
ax_corr.set_xticks([0, np.pi / 8, np.pi / 4, 3 * np.pi / 8, np.pi / 2],
                   ["0", "π/8", "π/4", "3π/8", "π/2"])
ax_corr.set_xlabel(r"tilt $\theta$ (Alice by $-\theta$, Bob by $+\theta$)")
ax_corr.set_ylabel("correlator E")
ax_corr.set_title("the two curves touch three times; only one of them is curved")
ax_corr.legend(fontsize=9)
fig_correlator.tight_layout()

The two models touch at exactly three tilts: $\theta = 0$, where the
computational-basis experiment of (a) lives and both are perfectly correlated;
$\theta = \pi/4$, where both are uncorrelated; and $\theta = \pi/2$, where both are
perfectly anti-correlated. Everywhere else the quantum correlation is *stronger* — the
straight classical line is the chord, the quantum cosine bows away from it. Rotating the
question is where the two states, indistinguishable in (a), come apart.

But a single model beaten by a curve proves nothing — maybe a cleverer hidden-variable
model catches up. Part (d) is the argument that closes that door for good.

### (d) CHSH: an exhaustive classical bound, and a quantum number above it

Four measurement settings, two for each side, combined into one score $S$ built from four
correlators. The classical side is not estimated: **every** deterministic local strategy is
enumerated, all 16 of them, and the best is reported. Any hidden-variable model whatsoever —
with any amount of shared randomness — is a probabilistic mixture of those 16, and $S$ is
linear in the mixing weights, so no mixture can beat the best pure strategy. The bound is a
theorem, not a measurement.

The settings are the polarizer angles of the real experiments: $0°, 22.5°, 45°, 67.5°$.
Note the factor of 2 in `Ry(theta=-2 * angle)`: an axis at angle $\alpha$ in the lab is a
rotation of $2\alpha$ on the Bloch sphere, because states $90°$ apart in the lab are
*orthogonal*, i.e. antipodal on the sphere. (`qsim.algorithms.chsh` quotes the same
settings already doubled, as Bloch angles; **[03 — Bell
tests](../03-bell-tests-teleportation.ipynb)** builds this from scratch.)

The quantum score is **sampled**, 20,000 shots per correlator, not read off the amplitudes.

In [ ]:
A_ANGLES = (0.0, np.pi / 4)                      # Alice:  0 deg, 45 deg
B_ANGLES = (np.pi / 8, 3 * np.pi / 8)            # Bob:   22.5 deg, 67.5 deg
# With these four settings the term carrying the minus sign is E(a, b') rather than the
# textbook's E(a', b'). Which one is subtracted is pure labelling -- Bob's 67.5 deg axis is
# the textbook's -22.5 deg axis turned end for end, which flips the sign of his answer.
# The classical enumeration below uses the identical combination, so the comparison is fair.
CHSH_SIGNS = (+1, -1, +1, +1)


def chsh_score(correlators) -> float:
    """Combine E(a,b), E(a,b'), E(a',b), E(a',b') into the CHSH score S."""
    return float(sum(sign * e for sign, e in zip(CHSH_SIGNS, correlators)))


def sampled_correlator(alpha: float, beta: float, shots: int, seed: int) -> float:
    """Estimate E from ``shots`` measurements of a freshly prepared Bell pair."""
    qc_chsh = Circuit(name="chsh", seed=seed)
    a_q, b_q = qc_chsh.alloc_many(2)
    H(a_q)
    CNOT(a_q, b_q)
    Ry(a_q, theta=-2.0 * alpha)
    Ry(b_q, theta=-2.0 * beta)
    counts = qc_chsh.inspect.sample(shots)
    # Outcome 0 is the answer +1 and outcome 1 is -1, so the product A*B is +1 when the
    # two bits agree and -1 when they differ.
    agree = counts["00"] + counts["11"]
    return (2 * agree - shots) / shots


settings = [(A_ANGLES[0], B_ANGLES[0]), (A_ANGLES[0], B_ANGLES[1]),
            (A_ANGLES[1], B_ANGLES[0]), (A_ANGLES[1], B_ANGLES[1])]
sampled = [sampled_correlator(a_ang, b_ang, 20_000, seed=100 + i)
           for i, (a_ang, b_ang) in enumerate(settings)]
s_quantum = chsh_score(sampled)

# Every deterministic local strategy: Alice's two answers and Bob's two, each fixed at +-1
# in advance. E(a, b) is then simply A*B, because nothing is random any more.
strategies = [((A, Ap, B, Bp), chsh_score((A * B, A * Bp, Ap * B, Ap * Bp)))
              for A, Ap, B, Bp in product((-1, 1), repeat=4)]
s_classical = max(score for _, score in strategies)

print("sampled correlators:", np.round(sampled, 4))
print(f"S_quantum (20,000 shots each) = {s_quantum:.4f}   [2*sqrt(2) = {2 * np.sqrt(2):.4f}]")
print(f"S_classical, exact over all {len(strategies)} deterministic strategies = {s_classical}")
assert s_quantum > 2.6
assert s_classical == 2.0

The left panel below is the entire classical theory of this experiment: sixteen bars, one
per strategy, nothing sampled and nothing left out. The right panel puts the three numbers
side by side — the exact classical ceiling, the sampled quantum score, and Tsirelson's
bound $2\sqrt2$, which is the most quantum mechanics itself allows.

That last number matters for the shape of the argument. Quantum mechanics does not simply
ignore the classical limit; it has a limit of its own, higher but finite. There are
imaginable theories that would score all the way to 4 without allowing faster-than-light
signalling, and nature does not use them. The 2-norm picks out $2\sqrt2$.

In [ ]:
fig_chsh, (ax_strat, ax_bar) = plt.subplots(1, 2, figsize=(10.5, 3.8),
                                            gridspec_kw={"width_ratios": [2, 1]})

scores = np.array([score for _, score in strategies])
ax_strat.bar(np.arange(len(scores)), scores,
             color=["#c33b53" if s == 2 else "#8a8f98" for s in scores])
ax_strat.axhline(2.0, color="#17797c", lw=1.4, ls="--")
ax_strat.axhline(s_quantum, color="#c33b53", lw=1.8)
ax_strat.annotate(f"quantum, sampled: {s_quantum:.3f}", (0.3, s_quantum + 0.12),
                  color="#c33b53", fontsize=9)
ax_strat.annotate("classical ceiling: 2", (0.3, 2.12), color="#17797c", fontsize=9)
ax_strat.set_xlabel("all 16 deterministic local strategies (A, A′, B, B′)")
ax_strat.set_ylabel("CHSH score S")
ax_strat.set_ylim(-3.4, 3.4)
ax_strat.set_title("the classical bound, enumerated not estimated")

ax_bar.bar([0, 1, 2], [s_classical, s_quantum, 2 * np.sqrt(2)],
           color=["#17797c", "#c33b53", "#8a8f98"])
ax_bar.set_xticks([0, 1, 2], ["classical\n(exact)", "quantum\n(sampled)",
                              "Tsirelson\n$2\\sqrt{2}$"])
ax_bar.set_ylim(0, 3.2)
ax_bar.set_title("S")
fig_chsh.tight_layout()

Every one of the 16 pre-agreed strategies scores at most 2, and the winners (red) reach
exactly 2. The sampled quantum value sits near $2\sqrt2 \approx 2.828$, above the ceiling
by far more than the sampling noise, which is about $0.007$ per correlator and so of
order $0.01$ in the combined score.

So the ignorance story of (b) is not merely unnecessary — it is *false*. No assignment of
definite answers-in-advance, and no mixture of such assignments, reproduces what the Bell
pair does. The identical statistics of (a) were a coincidence of one particular question.

**Being honest about what a simulator shows.** This demonstrates the *statistics*. The
physics that makes a real Bell test compelling — spacelike separation of the two
measurements, random setting choices made too late to influence the other side, closing the
detection loophole — is exactly the part a simulator cannot supply. Those experiments have
been done (Nobel Prize, 2022); this cell only shows that the numbers are what they are.

## Part 5 — The bridge: decoherence squares your unitaries

Take a unitary $U$ and square every entry's magnitude: $S_{ij} = \lvert U_{ij}\rvert^2$.
Unitarity says the rows and columns of $U$ are unit vectors, which says exactly that the
rows and columns of $S$ sum to 1 — so $S$ is a **doubly stochastic** matrix, a legal `csim`
gate.

The proof is one line each way. Row $j$ of $U$ is a unit vector, so
$\sum_i \lvert U_{ji}\rvert^2 = 1$: the rows of $S$ sum to 1. The columns of $U$ are unit
vectors too, so the columns of $S$ sum to 1 as well. Every entry is a squared magnitude and
therefore non-negative. That is the definition of doubly stochastic.

"Doubly" matters. An arbitrary stochastic matrix only needs its *columns* to sum to 1;
requiring the rows to sum to 1 as well is the extra condition that says the map has no
preferred destination — it leaves the uniform distribution alone. By the Birkhoff–von
Neumann theorem, every doubly stochastic matrix is a convex mixture of permutation
matrices: a random choice among the reversible classical maps of Part 1. So squaring a
unitary lands you not on any classical dynamics, but specifically on "pick a permutation at
random" — reversible classical physics with the randomness put in by hand.

**The claim.** This is not a formal pun. $S$ is the dynamics you actually get when full
dephasing follows every gate. The classical simulator is inside the quantum one, exactly,
and decoherence is the projection.

### (a) The formal check

In [ ]:
def haar_unitary(generator: np.random.Generator, d: int, n: int) -> np.ndarray:
    """n unitaries drawn uniformly (Haar measure), by the standard Mezzadri QR recipe."""
    z = (generator.normal(size=(n, d, d)) + 1j * generator.normal(size=(n, d, d)))
    # QR factors z into (unitary Q) @ (upper-triangular R). Q alone is *not* Haar-uniform
    # because QR is only unique up to phases on the diagonal of R; dividing those phases
    # out is the correction that makes the distribution uniform.
    q_mat, r_mat = np.linalg.qr(z)
    phases = np.diagonal(r_mat, axis1=1, axis2=2)
    return q_mat * (phases / np.abs(phases))[:, None, :]


worst_ds = 0.0
for d in (2, 4):
    unitaries = haar_unitary(rng, d, 500)
    squared_u = np.abs(unitaries) ** 2
    row_err = np.abs(squared_u.sum(axis=2) - 1.0).max()
    col_err = np.abs(squared_u.sum(axis=1) - 1.0).max()
    worst_ds = max(worst_ds, row_err, col_err, -squared_u.min())
    print(f"SU({d}), 500 Haar samples: rows off by <= {row_err:.2e}, "
          f"columns off by <= {col_err:.2e}")
assert worst_ds < 1e-12

### (b) The operational check

Now stop doing algebra and run the experiment. One qubit. Repeat: apply $R_x(0.9)$, then let
a **fresh** environment qubit look at it with `dephasing_coupling(..., theta=pi)` — a
perfect record of whether the qubit is $\lvert 0\rangle$ or $\lvert 1\rangle$.

Nothing is traced out, nothing is measured, nothing random happens. The environment qubits
stay in the state tensor and the global state stays perfectly pure forever
(**[decoherence_dial](decoherence_dial.ipynb)** is the exhibit for that point). A fresh
environment qubit per step means the circuit grows to 7 qubits by step 6 — well within
budget, and the freshness matters: an environment that could be re-used could be un-learned.

Against that, run `csim` with $S = \lvert R_x(0.9)\rvert^2$ and compare. The qubit's own
distribution is read two ways: from the diagonal of its reduced density matrix, and by
honestly marginalizing the full $2^n$ outcome distribution down onto the qubit's axis —
`csim`'s `marginal`, applied to the quantum simulator's probabilities.

In [ ]:
THETA_U = 0.9
S_SQUARED = np.abs(np.array([[np.cos(THETA_U / 2), -1j * np.sin(THETA_U / 2)],
                             [-1j * np.sin(THETA_U / 2), np.cos(THETA_U / 2)]])) ** 2
print("S = |Rx(0.9)|^2 =\n", S_SQUARED)
print("and that is exactly LAZY(sin^2(theta/2)) from Part 1:\n",
      LAZY(np.sin(THETA_U / 2) ** 2))
assert np.allclose(S_SQUARED, LAZY(np.sin(THETA_U / 2) ** 2), atol=1e-15)

qc_bridge = Circuit(name="bridge", seed=5)
q_bridge = qc_bridge.alloc("q")
p_bridge = cstate(1)                             # csim's copy of the same experiment
worst_bridge = 0.0

for k in range(1, 7):
    Rx(q_bridge, theta=THETA_U)
    env = qc_bridge.environment(1)               # a fresh recorder every step
    dephasing_coupling(q_bridge, env[0], theta=np.pi)
    p_bridge = apply_stochastic(p_bridge, S_SQUARED, 0)

    from_rho = np.real(np.diag(qc_bridge.inspect.reduced_density_matrix([q_bridge])))
    # The honest marginal: reshape the flat probability vector back to (2,)*n -- axis 0 is
    # the system qubit, allocated first -- and sum away every environment axis. This is
    # csim's own `marginal`, run on quantum probabilities.
    full = qc_bridge.inspect.probabilities().reshape((2,) * qc_bridge.n_qubits)
    from_marginal = marginal(full, [0])

    worst_bridge = max(worst_bridge, np.abs(from_rho - p_bridge).max(),
                       np.abs(from_marginal - p_bridge).max())
    print(f"k={k}  csim S^k p0 = {p_bridge}   quantum marginal = {from_marginal}"
          f"   |diff| = {np.abs(from_marginal - p_bridge).max():.2e}")

print(f"\ncircuit is now {qc_bridge.n_qubits} qubits; global state still pure: "
      f"S = {qc_bridge.inspect.entanglement_entropy(list(qc_bridge.qubits)):.2e} bits")
assert worst_bridge < 1e-10

Agreement to $10^{-16}$, and the matrix $S = \lvert R_x(0.9)\rvert^2$ turned out to be
`LAZY(0.189)` — the very channel whose one-way crawl we watched in Part 1c. The quantum
simulator, run through a fully decohering environment, *is* the classical simulator with
the squared matrix. Not an approximation of it; the same numbers.

Two details are load-bearing.

**The basis is not free.** `dephasing_coupling` lets the environment ask "is this qubit
$\lvert 0\rangle$ or $\lvert 1\rangle$?" — a question in the computational basis — and it
is precisely the populations *in that basis* that survive while everything else dies. A
different question would preserve a different set of numbers and hand back a different
classical theory. Which basis the world happens to monitor is what picks out which
quantities look classical, and that is **einselection**; the exhibit for it is
**[einselection](einselection.ipynb)**.

**Nothing irreversible happened.** The last line reports the entropy of the whole 7-qubit
state as $10^{-15}$ bits: still pure, still a vector, still exactly invertible. The
classical behaviour is what the *system's* description does when the environment's records
are left out of the account. Put the records back in — as **[quantum
eraser](quantum_eraser.ipynb)** does — and the interference returns intact. The classical
world is not created here; it is a view.

### (c) The dial

Full dephasing is one end of a knob. `dephasing_coupling(..., theta=t)` lets the
environment learn a controllable *amount*: at $t = 0$ it learns nothing, at $t = \pi$
everything, and coherence falls off as $\cos(t/2)$ in between. Sweep $t$ and plot the
qubit's Bloch vector as the same $R_x$ orbit repeats 10 times.

$R_x$ moves the Bloch vector in the $y$–$z$ plane and dephasing shrinks $x$ and $y$, so the
whole story fits in that plane.

In [ ]:
DIAL_STEPS = 10


def dial_trajectory(theta_env: float) -> np.ndarray:
    """Bloch vectors of one qubit under (Rx, then dephase by theta_env), 10 times."""
    qc_dial = Circuit(name="dial", seed=5)
    q_dial = qc_dial.alloc("q")
    track = [qc_dial.inspect.bloch_vector(q_dial)]
    for _ in range(DIAL_STEPS):
        Rx(q_dial, theta=THETA_U)
        env_dial = qc_dial.environment(1)
        dephasing_coupling(q_dial, env_dial[0], theta=theta_env)
        track.append(qc_dial.inspect.bloch_vector(q_dial))
    return np.array(track)


dial_thetas = np.linspace(0.0, np.pi, 7)
trajectories = [dial_trajectory(float(t)) for t in dial_thetas]

norm_at_zero = np.linalg.norm(trajectories[0], axis=1)
norm_at_pi = np.linalg.norm(trajectories[-1][-1])
print(f"theta = 0:   Bloch length constant to {np.abs(norm_at_zero - 1).max():.2e}")
print(f"theta = pi:  Bloch length after {DIAL_STEPS} steps = {norm_at_pi:.4f}")
assert np.abs(norm_at_zero - 1).max() < 1e-12
assert norm_at_pi < 0.02

Seven settings of the knob, ten steps each, all in the $y$–$z$ plane of the Bloch ball. The
grey circle is the surface — the pure states — and the cross at the centre is the maximally
mixed state, which is to say a fair classical coin.

Read the figure as a family of paths between the two halves of Part 1c: the outer circle is
the orbit, the vertical axis is the crawl, and the knob interpolates. Each step turns the
qubit by a substantial 0.9 radians, so the dots are far apart and the straight segments
joining them are only guides — the true motion between two dots is an arc.

In [ ]:
fig_dial, ax_dial = plt.subplots(figsize=(6.4, 6.0))
ax_dial.plot(np.cos(circle), np.sin(circle), color="#8a8f98", lw=1.0)
colors = plt.get_cmap("plasma")(np.linspace(0.05, 0.85, len(dial_thetas)))

for theta_env, track, color in zip(dial_thetas, trajectories, colors):
    ax_dial.plot(track[:, 1], track[:, 2], "-o", ms=3.4, lw=1.5, color=color,
                 label=f"θ = {theta_env / np.pi:.2f}π")
ax_dial.scatter([0], [0], marker="+", s=130, color="#17797c", zorder=5,
                label="maximally mixed:\na classical coin")
ax_dial.annotate(r"start $|0\rangle$", (0, 1), fontsize=9,
                 textcoords="offset points", xytext=(8, -2))
ax_dial.set_xlabel("Bloch y")
ax_dial.set_ylabel("Bloch z")
ax_dial.set_aspect("equal")
ax_dial.set_xlim(-1.15, 1.35)
ax_dial.set_ylim(-1.15, 1.15)
ax_dial.set_title("one knob from quantum to classical\n"
                  r"10 × [$R_x(0.9)$, then let the world watch by $\theta$]")
ax_dial.legend(fontsize=8, loc="lower right")
fig_dial.tight_layout()

One figure with the whole notebook in it.

At $\theta = 0$ the trajectory rides the unit circle: Part 1c's orbit, norm 1 forever,
reversible, quantum. At $\theta = \pi$ it collapses onto the vertical axis at the first
step — all coherence gone — and then crawls down that axis toward the centre, and *that
line is Part 1c's classical segment*, drawn vertically, with `LAZY(0.189)` doing the
crawling. Every curve in between is a spiral: partially quantum, partially classical, no
sharp boundary anywhere.

The classical world is not a different theory sitting next to the quantum one. It is the
quantum theory at one end of that knob.

## Closing: the dictionary, and its one-way asymmetry

| | classical | quantum |
|---|---|---|
| norm | 1-norm | 2-norm |
| state space | simplex (a segment for one bit) | Bloch ball |
| reversible dynamics | permutations only — finite | unitaries — a continuous group |
| combining alternatives | probabilities add | amplitudes add, and can cancel |
| shared preparation | correlation = ignorance of a definite fact | entanglement = correlation under full knowledge |
| $S(\text{whole})$ vs $S(\text{part})$ | $\ge$, always | can be $0 < 1$ |
| pure states | the corners of the simplex | the whole surface of the ball |

The two columns are not rival theories. **The classical column is the quantum column as
seen by an environment that records everything.** The environment's records square the
amplitudes — that is Part 5, and squaring amplitudes is the Born rule this notebook has
been using since its first cell.

And the relationship runs one way only:

- The embedding of classical *into* quantum is **exact**. Part 5: full dephasing after
  every gate turns the quantum simulator into `csim` with $S = \lvert U\rvert^2$, to
  $10^{-16}$.
- The attempted embedding of quantum into classical **fails**, at three specific places:
  **Part 1** (there is no continuous reversibility on a simplex — the square root of NOT
  does not exist), **Part 3** (there is no cancellation — probabilities only add), and
  **Part 4** (there is no purity with mixed parts — no ignorance model reaches $2\sqrt2$).

Those three failures have names: no continuous reversibility, no interference, no
entanglement. They are the subject of the rest of this library.

### Where to go next

- **[decoherence_dial](decoherence_dial.ipynb)** (B1): Part 5c's knob, studied on its own —
  what you lose in visibility is exactly what the environment gained in
  distinguishability.
- **[entanglement_and_marginals](entanglement_and_marginals.ipynb)** (A2): Part 4b's
  reduced density matrices, in detail.
- **[quaternions_and_spin](quaternions_and_spin.ipynb)** (A4): why Part 1's rotations come
  with half-angles, and what else lives in $SU(2)$.
- **[03 — Bell tests and teleportation](../03-bell-tests-teleportation.ipynb)** and
  `qsim.algorithms.chsh`: Part 4d built properly, with the correlators derived rather than
  sampled.
- **[05 — interferometers](../05-interferometers.ipynb)**: Part 3's cancellation, turned
  into apparatus.
- `gleason_teaser.ipynb` (E3, not yet built) will take up the question Part 1c raised — why
  the 2-norm, and not some other exponent.

## Assertions

Every claim above, re-checked in one place. If any of this rots, the notebook fails loudly
under `jupyter execute`.

In [ ]:
# --- Part 0: the two simulators are the same code, and both conserve their norm ---
# This assertion deliberately couples the notebook to qsim's source. If the kernel is ever
# rewritten, this is a cell to re-read, not to delete.
assert normalized_body(qsim.state.apply_1q, {"psi": "state", "u": "matrix"}) == \
    normalized_body(apply_stochastic, {"p": "state", "s": "matrix"})
assert worst_l1 < 1e-12, "1-norm not preserved by stochastic maps"
assert worst_l2 < 1e-12, "2-norm not preserved by unitaries"
assert p_classical.min() >= -1e-15, "a probability went negative"

# --- Part 1: rigidity, no sqrt(NOT) classically, crawl vs orbit ---
for d_check in (3, 4):
    distances_check, survivors_check = search[d_check]
    assert survivors_check.sum() > 0, "the search found nothing to test"
    assert distances_check[survivors_check].max() < 1e-6, "a reversible non-permutation!"
assert residual_flip.min() > 0.05, "a classical square root of NOT"
assert abs(residual_coin.min()) < 1e-12 and abs(grid[best_coin[0]] - 0.5) < 1e-12
for start_bit in (0, 1):
    qc_sx_check = Circuit(seed=1)
    q_sx_check = qc_sx_check.alloc()
    qc_x_check = Circuit(seed=1)
    q_x_check = qc_x_check.alloc()
    if start_bit == 1:
        X(q_sx_check)
        X(q_x_check)
    SX(q_sx_check)
    SX(q_sx_check)
    X(q_x_check)
    assert np.abs(qc_sx_check.inspect.state_vector()
                  - qc_x_check.inspect.state_vector()).max() < 1e-12, "SX^2 != X"
assert abs(abs(qc_half.inspect.amplitude("1")) - 1.0) < 1e-12, "Rx(pi/2)^2 != X"
assert np.abs(np.array(bloch_norms) - 1).max() < 1e-12, "Rx changed the Bloch length"
assert np.all(np.diff(lazy_l1) <= 1e-15), "LAZY was not monotone toward uniform"
assert np.abs(root @ root - unitary).max() < 1e-12, "sqrt of a unitary failed"

# --- Part 2: the embedding is an isometry onto the octant; its dynamics is not linear ---
assert worst_embed < 1e-12, "sqrt(p) is not a unit vector, or overlap != Bhattacharyya"
assert np.abs(np.linalg.norm(embedded, axis=1) - 1).max() < 1e-12
assert max(perm_residuals) < 1e-10, "a permutation failed to be linear in sqrt(p)"
assert min(random_residuals) > 0.01, "a non-permutation was linear in sqrt(p)"

# --- Part 3: paths add, or cancel ---
assert np.abs(classical_p0 - 0.5).max() < 1e-12, "a classical phase did something"
assert np.abs(quantum_p0 - np.cos(phis / 2) ** 2).max() < 1e-12, "not cos^2(phi/2)"
assert abs(via_0 - 0.5) < 1e-12 and abs(via_1 + 0.5) < 1e-12, "path amplitudes not +-1/2"
assert abs(probs_bomb[2] - 0.25) < 1e-12, "Elitzur-Vaidman detection probability"

# --- Part 4: same statistics, different purity, and no local model reaches 2.828 ---
assert np.allclose(p_pair.reshape(-1), qc_bell.inspect.probabilities(), atol=1e-12)
assert np.allclose(marginal(p_pair, [0]), [0.5, 0.5], atol=1e-12)
assert np.allclose(qc_bell.inspect.reduced_density_matrix([alice]), np.eye(2) / 2,
                   atol=1e-12)
assert abs(s_joint_c - 1.0) < 1e-12 and abs(s_marginal_c - 1.0) < 1e-12
assert abs(s_joint_q) < 1e-12 and abs(s_marginal_q - 1.0) < 1e-12
assert s_joint_q < s_marginal_q - 0.5, "the whole was not more certain than its part"
assert np.abs(quantum_E - np.cos(2 * thetas)).max() < 1e-12
assert np.all(np.abs(quantum_E) >= np.abs(lhv_exact) - 1e-12), "the LHV model kept up"
assert abs(quantum_E[15]) > abs(lhv_exact[15]) + 0.1
assert s_quantum > 2.6, "sampled CHSH score too low"
assert s_classical == 2.0, "exhaustive classical CHSH bound is not 2"
assert len(strategies) == 16

# --- Part 5: |U|^2 is doubly stochastic, and it is the dynamics you actually get ---
assert worst_ds < 1e-12, "|U|^2 was not doubly stochastic"
assert np.allclose(S_SQUARED, LAZY(np.sin(THETA_U / 2) ** 2), atol=1e-15)
assert worst_bridge < 1e-10, "dephased quantum populations != S^k p0"
assert np.abs(np.linalg.norm(trajectories[0], axis=1) - 1).max() < 1e-12
assert np.linalg.norm(trajectories[-1][-1]) < 0.02

print("all assertions passed")